# Triton 学习全流程 — 一键运行所有示例

**使用前请确保：** Runtime → Change runtime type → GPU (T4 或更好)

本 notebook 会：
1. 挂载 Google Drive
2. 安装 Triton + 检查环境
3. 依次运行 6 个 Triton 学习示例（从基础到进阶）

| Step | 文件 | 核心概念 |
|------|------|----------|
| 1 | `01_vector_add.py` | Triton 编程模型基础 |
| 2 | `02_softmax.py` | Kernel Fusion + Reduction |
| 3 | `03_matmul.py` | GEMM + Auto-tune + Tensor Core |
| 4 | `04_flash_attention.py` | Flash Attention (Online Softmax) |
| 5 | `05_layernorm.py` | LayerNorm Forward + Backward |
| 6 | `06_quantize_matmul.py` | INT8 量化矩阵乘法 |

---
## 0. 环境准备

In [1]:
# 挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 检查 GPU
!nvidia-smi

Mon Apr  6 11:16:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# 安装 triton（Colab 已预装 torch，只需装 triton）
!pip install triton -q

In [4]:
import torch
import triton
print(f"PyTorch: {torch.__version__}")
print(f"Triton:  {triton.__version__}")
print(f"CUDA:    {torch.version.cuda}")
print(f"GPU:     {torch.cuda.get_device_name(0)}")
print(f"显存:    {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

PyTorch: 2.10.0+cu128
Triton:  3.6.0
CUDA:    12.8
GPU:     Tesla T4
显存:    14.6 GB


In [5]:
import os
TRITON_DIR = "/content/drive/MyDrive/cuda/triton_samples"
os.chdir(TRITON_DIR)
print(f"工作目录: {os.getcwd()}")
print(f"文件列表:")
!ls -la *.py

工作目录: /content/drive/MyDrive/cuda/triton_samples
文件列表:
-rw------- 1 root root 3825 Apr  6 10:27 01_vector_add.py
-rw------- 1 root root 4597 Apr  6 10:27 02_softmax.py
-rw------- 1 root root 8012 Apr  6 10:28 03_matmul.py
-rw------- 1 root root 8921 Apr  6 10:29 04_flash_attention.py
-rw------- 1 root root 8538 Apr  6 10:30 05_layernorm.py
-rw------- 1 root root 8813 Apr  6 10:31 06_quantize_matmul.py


---
## Step 1: Vector Add — Triton Hello World

对标 CUDA `vector_add.cu`，理解 Triton 最基本的编程模型：
- `@triton.jit` 定义 kernel
- `tl.program_id` ≈ `blockIdx.x`
- `tl.arange` ≈ `threadIdx.x`
- `tl.load / tl.store` + `mask` 处理边界

In [15]:
%run 01_vector_add.py

Step 1: Vector Add — Triton Hello World

--- 功能测试 ---
Input size: torch.Size([1024])
First 5 results: [-0.12534070014953613, 0.1870027780532837, -1.5071995258331299, 0.13156071305274963, 1.7027372121810913]
Correct: True

--- 性能对比 ---
           N | PyTorch (ms) |  Triton (ms) |    Ratio
-------------------------------------------------------
      65,536 |       0.0100 |       0.0094 |   0.94x
     262,144 |       0.0179 |       0.0172 |   0.97x
   1,048,576 |       0.0572 |       0.0572 |   1.00x
   4,194,304 |       0.2155 |       0.2159 |   1.00x
  16,777,216 |       0.8480 |       0.8475 |   1.00x
  67,108,864 |       3.3605 |       3.3573 |   1.00x

* Ratio = Triton/PyTorch，越接近 1.0 越好，>1 表示 Triton 更慢
* vector_add 是 memory-bound 操作，两者最终都受限于显存带宽，差距应该很小


---
## Step 2: Fused Softmax — Kernel Fusion 入门

核心概念：把 max → sub → exp → sum → div 五个操作融合到一个 kernel
- PyTorch: 5 次 kernel launch + 大量 global memory 读写
- Triton: 1 次 launch，数据留在寄存器中完成所有计算

In [16]:
%run 02_softmax.py

Step 2: Fused Softmax

--- 功能测试 ---
Input shape: torch.Size([4, 8])
Output row sum (should be ~1.0): [0.9999999403953552, 1.0, 1.0, 0.9999998807907104]
Max diff vs PyTorch: 5.96e-08

--- 性能对比 ---
           Shape | PyTorch (ms) |  Triton (ms) |    Ratio
------------------------------------------------------------
       1024x1024 |       0.0439 |       0.0392 |   0.89x
       2048x2048 |       0.1592 |       0.1471 |   0.92x
       4096x4096 |       0.5562 |       0.5889 |   1.06x
       8192x4096 |       1.0979 |       1.1742 |   1.07x
       4096x8192 |       1.1388 |       1.1805 |   1.04x

* Ratio = Triton/PyTorch，<1 表示 Triton 更快
* PyTorch 调用 cuDNN 优化实现，Triton 的价值在于可定制 fusion


---
## Step 3: Matrix Multiplication (GEMM)

对标整个 `matmul_optimize/` 系列。Triton 用 ~80 行覆盖了 CUDA 7 个 step 的优化：

| CUDA 手写优化 | Triton 对应 |
|--------------|------------|
| shared memory tiling | `tl.load` 循环（编译器自动） |
| 寄存器 tiling | `BLOCK_SIZE_M/N/K` 参数 |
| bank conflict | 编译器自动处理 |
| double buffering | 编译器自动 prefetch |
| Tensor Core | `tl.dot()` 自动使用 |
| 手动调参 | `@triton.autotune` 自动搜索 |

In [17]:
%run 03_matmul.py

Step 3: Matrix Multiplication (GEMM)

--- 功能测试 ---
A: torch.Size([64, 128]), B: torch.Size([128, 32]) → C: torch.Size([64, 32])
Max diff vs cuBLAS: 1.95e-03

--- 性能对比 (FP16) ---
   M=N=K |  cuBLAS (ms) |  Triton (ms) | TFLOPS (Triton) | Triton/cuBLAS
---------------------------------------------------------------------------
     512 |       0.0184 |       0.1704 |           1.58T |        10.8%
    1024 |       0.0594 |       1.8716 |           1.15T |         3.2%
    2048 |       0.4815 |      16.8702 |           1.02T |         2.9%
    4096 |       6.4868 |     147.7059 |           0.93T |         4.4%

* Triton/cuBLAS 表示达到 cuBLAS 性能的百分比，>80% 就不错
* cuBLAS 经过数十年优化，Triton 用 ~80 行 Python 达到这个比例已经很强


---
## Step 4: Flash Attention — Triton 杀手级应用

Flash Attention 核心思想：
- 不存储 N×N attention 矩阵（O(N²) → O(N) 内存）
- 分块遍历 K/V，在线更新 softmax 统计量
- QK^T → scale → mask → softmax → @V 全部 fused

手写 CUDA ~2000 行，Triton ~120 行

In [18]:
%run 04_flash_attention.py

Step 4: Flash Attention

--- 功能测试 ---
Shape: Q/K/V = [2, 4, 128, 64]
Output shape: torch.Size([2, 4, 128, 64])
Max diff vs reference: 1.95e-03

--- 性能对比 (B=4, H=32, D=64, causal=True) ---
  SeqLen | PyTorch SDPA (ms) | Triton FA (ms) |  Speedup
------------------------------------------------------------
     256 |            0.1123 |        15.4783 |    0.01x
     512 |            0.3634 |        55.4298 |    0.01x
    1024 |            1.4515 |       208.4598 |    0.01x
    2048 |            5.7919 |       809.8271 |    0.01x
    4096 |           22.0785 |      3186.1841 |    0.01x


---
## Step 5: Layer Normalization — Forward + Backward

LLM 推理中频率最高的算子之一：
- Forward: fused mean → var → normalize → scale → bias
- Backward: fused 梯度计算
- Memory-bound 算子，fusion 收益主要来自减少 global memory traffic

In [19]:
%run 05_layernorm.py

Step 5: Layer Normalization

--- Forward 功能测试 ---
Input: [128, 768]
Max diff vs PyTorch: 5.64e-03
Output mean (should be ~0): 0.0000
Output std  (should be ~1): 0.9998

--- Backward 功能测试 ---
dx max diff: 1.43e-06
dw max diff: 5.72e-06
db max diff: 5.72e-06

--- 性能对比 (Forward) ---
           Shape | PyTorch (ms) |  Triton (ms) |  Speedup
------------------------------------------------------------
        1024x768 |       0.0323 |       0.0310 |    1.04x
       1024x1024 |       0.0424 |       0.0403 |    1.05x
       2048x2048 |       0.1906 |       0.1504 |    1.27x
       4096x4096 |       0.8189 |       0.5986 |    1.37x
       8192x4096 |       1.6226 |       1.1829 |    1.37x


---
## Step 6: Quantized MatMul (INT8) — LLM 推理优化

大模型推理核心优化：
- 权重 INT8 存储（省显存 + 带宽），激活保持 FP16
- Dequantize 在 kernel 内 fused 完成，零额外开销
- 小 batch (decode) 场景收益最大（memory-bound）

In [20]:
%run 06_quantize_matmul.py

Step 6: Quantized MatMul (Weight-Only INT8)

--- 功能测试 ---
原始权重: torch.Size([256, 128]) (64.0 KB)
量化权重: torch.Size([256, 128]) (32.0 KB)
压缩率: 2x
Output shape: torch.Size([64, 128])
Max diff vs dequantized reference: 3.12e-02

--- 性能对比 (FP16 cuBLAS vs INT8 Triton) ---
               M×K×N |  FP16 (ms) | INT8 Triton (ms) |  Speedup |   Max Diff
------------------------------------------------------------------------------
         1×4096×4096 |     0.1609 |           1.1897 |    0.14x |   1.25e-01
        32×4096×4096 |     0.1457 |           1.2385 |    0.12x |   2.50e-01
       128×4096×4096 |     0.1612 |           3.3686 |    0.05x |   2.50e-01
        1×4096×11008 |     0.4904 |           4.7281 |    0.10x |   1.25e-01
       32×4096×11008 |     0.3917 |           5.0586 |    0.08x |   2.50e-01
      128×4096×11008 |     0.4953 |          10.2272 |    0.05x |   2.50e-01

注意：INT8 量化的加速主要来自减少 memory bandwidth，
在小 M (decode) 场景收益最大（memory-bound），大 M 收益递减（趋向 compute-bound）。


### 验证性能：排除编译干扰 (Warmup)
我们可以选取 `02_softmax.py` 为例，手动进行多次迭代，观察在稳定状态下的速度对比。

In [21]:
import torch
import triton
import os

# 确保在正确的目录下
os.chdir('/content/drive/MyDrive/cuda/triton_samples')

# 模拟一个多次运行的环境来观察性能
print("正在进行性能复测 (包含 Warmup)... ")
# 重新运行 softmax 示例，注意观察第二次及以后的输出（如果脚本内部支持多次迭代）
%run 02_softmax.py

print("\n提示：在生产环境中，Triton 的核心价值是让你能用 Python 写出接近 CUDA 性能的自定义算子，而不是在所有基础算子上都超越 cuBLAS。")

正在进行性能复测 (包含 Warmup)... 
Step 2: Fused Softmax

--- 功能测试 ---
Input shape: torch.Size([4, 8])
Output row sum (should be ~1.0): [1.0000001192092896, 1.0, 0.9999998807907104, 1.0000001192092896]
Max diff vs PyTorch: 5.96e-08

--- 性能对比 ---
           Shape | PyTorch (ms) |  Triton (ms) |    Ratio
------------------------------------------------------------
       1024x1024 |       0.0440 |       0.0398 |   0.91x
       2048x2048 |       0.1617 |       0.1470 |   0.91x
       4096x4096 |       0.5552 |       0.5887 |   1.06x
       8192x4096 |       1.0971 |       1.1722 |   1.07x
       4096x8192 |       1.1371 |       1.1787 |   1.04x

* Ratio = Triton/PyTorch，<1 表示 Triton 更快
* PyTorch 调用 cuDNN 优化实现，Triton 的价值在于可定制 fusion

提示：在生产环境中，Triton 的核心价值是让你能用 Python 写出接近 CUDA 性能的自定义算子，而不是在所有基础算子上都超越 cuBLAS。


---
## 汇总

把上面的结果填入 `TRITON_ROADMAP.md` 的性能记录表中，和你的 CUDA 实现做对比。

**核心收获**：
1. Triton 以 **block (program)** 为编程单位，不是 thread
2. shared memory、bank conflict、向量化、Tensor Core 等底层优化由**编译器自动完成**
3. `@triton.autotune` 自动搜索最优超参数
4. Kernel Fusion 是 Triton 最大的优势场景（softmax、attention、layernorm）
5. 量化推理是 Triton 在 LLM 生态中的核心应用